# Part 3: Tuning + Comparison — Picking the Winner
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Thursday — The Final Comparison

Sarah has three contenders trained on the same data:
- **L03 baseline** — logistic regression
- **Random Forest** — with `class_weight='balanced'`
- **Gradient Boosting** — also with `class_weight='balanced'`

Defaults gave the trees a slight edge. Today she runs proper hyperparameter tuning with `GridSearchCV` on both, then picks a final winner with honest cross-validated evidence.

**By the end of this notebook you will be able to:**
- Set up a `GridSearchCV` over a tree-based pipeline
- Read the best-params output and explain why those settings won
- Compare all three models on a final held-out test set
- Write Sarah's one-page recommendation to Marcus

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (11, 4.5)

print("✅ Libraries loaded — GridSearchCV ready")

## Step 1 — Setup (same data + preprocessor)

In [ ]:
df = pd.read_csv("data/northstar_churn.csv")
y  = df["churned"]
X  = df.drop(columns=["customer_id", "churned"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42,
)

numeric_features = ["age", "tenure_months", "num_purchases_quarter",
                    "avg_monthly_spend_gbp", "returns_per_purchase",
                    "last_login_days_ago", "avg_review_polarity",
                    "support_tickets_quarter"]
categorical_features = ["region", "subscription_tier"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("scl", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
                      categorical_features),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {len(X_train):,} customers · Test: {len(X_test):,} customers")

## Step 2 — GridSearch over Random Forest

We tune two of the most impactful hyperparameters: `n_estimators` and `min_samples_leaf`. (Keeping the grid small so it runs in under a minute on a typical laptop.)

In [ ]:
rf_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", n_jobs=-1, random_state=42)),
])

rf_grid = {
    "model__n_estimators":     [100, 200, 400],
    "model__min_samples_leaf": [1, 5, 20],
    "model__max_depth":        [None, 10],
}
# 3 × 3 × 2 = 18 combos × 5-fold = 90 model fits

start = time.time()
rf_search = GridSearchCV(
    rf_pipeline,
    rf_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,
)
rf_search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"GridSearchCV finished in {elapsed:.1f}s — fit {len(rf_search.cv_results_['params'])} configurations")
print()
print(f"Best CV F1: {rf_search.best_score_:.3f}")
print(f"Best params:")
for k, v in rf_search.best_params_.items():
    print(f"  · {k.replace('model__', ''):20s} = {v}")

## ⏸️ Pause and Predict

Before running the next cell, predict:

- For Gradient Boosting, which hyperparameter do you think will matter MORE — `learning_rate` or `max_iter`?
- What's a sensible range for `max_depth` of GB trees? (Hint from NB 03: shallow trees won.)

> *Sample expected:* `learning_rate` matters more than `max_iter` because they're coupled (lower lr can always be compensated with more iterations, but the dynamics differ). For `max_depth`, 3–8 is the typical sweet spot. Full depth often hurts.

In [ ]:
gb_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", random_state=42)),
])

gb_grid = {
    "model__learning_rate": [0.05, 0.1, 0.2],
    "model__max_iter":      [100, 200, 400],
    "model__max_depth":     [4, 6, None],
}
# 3 × 3 × 3 = 27 combos × 5-fold = 135 model fits

start = time.time()
gb_search = GridSearchCV(
    gb_pipeline,
    gb_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    refit=True,
)
gb_search.fit(X_train, y_train)
elapsed = time.time() - start

print(f"GridSearchCV finished in {elapsed:.1f}s — fit {len(gb_search.cv_results_['params'])} configurations")
print()
print(f"Best CV F1: {gb_search.best_score_:.3f}")
print(f"Best params:")
for k, v in gb_search.best_params_.items():
    print(f"  · {k.replace('model__', ''):20s} = {v}")

## Step 4 — The top-5 configurations side by side

Let's see the top 5 hyperparameter combinations for each model and how stable the F1 is across the grid.

In [ ]:
def top5(search, name):
    df_ = pd.DataFrame(search.cv_results_)
    df_ = df_.sort_values("mean_test_score", ascending=False).head(5)
    rows = []
    for _, row in df_.iterrows():
        params = ", ".join(f"{k.replace('model__','')}={v}" for k, v in row["params"].items())
        rows.append((row["mean_test_score"], row["std_test_score"], params))
    out = pd.DataFrame(rows, columns=[f"{name} CV F1", "std", "params"])
    return out

print("Top 5 Random Forest configurations:")
print(top5(rf_search, "RF").to_string(index=False))
print()
print("Top 5 Gradient Boosting configurations:")
print(top5(gb_search, "GB").to_string(index=False))

### 💡 What you should notice

- The **top configurations cluster tightly** — within ~0.01 F1 of each other. This is typical and tells you the model is "robust" — small parameter changes don't catastrophically change performance.
- **The winning hyperparameters** are usually the ones you'd guess from theory (shallow trees + lower learning rates for GB; moderate depth + many trees for RF).

## Step 5 — Train an L03 baseline for comparison

Re-train logistic regression on the same data so the comparison is clean.

In [ ]:
lr_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])

cv_f1_lr = cross_val_score(lr_pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
lr_pipeline.fit(X_train, y_train)

print(f"L03 LogisticRegression (with class_weight='balanced')  CV F1: {cv_f1_lr.mean():.3f} ± {cv_f1_lr.std():.3f}")

## Step 6 — Final comparison on the held-out test set

For each of the three models, predict on the test set at threshold 0.5 (the default) and at threshold 0.25 (the capacity-based threshold from L03).

In [ ]:
def evaluate(model, name, X_test, y_test):
    proba = model.predict_proba(X_test)[:, 1]
    rows = []
    for t in [0.5, 0.25]:
        pred = (proba >= t).astype(int)
        rows.append({
            "Model":     name,
            "Threshold": t,
            "Accuracy":  accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall":    recall_score(y_test, pred),
            "F1":        f1_score(y_test, pred),
            "Flagged":   int(pred.sum()),
        })
    return rows

all_results = []
all_results.extend(evaluate(lr_pipeline,           "LR (L03)",     X_test, y_test))
all_results.extend(evaluate(rf_search.best_estimator_, "RF (tuned)", X_test, y_test))
all_results.extend(evaluate(gb_search.best_estimator_, "GB (tuned)", X_test, y_test))

comparison_df = pd.DataFrame(all_results)
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

### 💡 What you should notice

- **At threshold 0.5**, GB (tuned) typically has the highest F1.
- **At threshold 0.25**, all three are closer together — but GB usually still leads.
- **Recall climbs as threshold falls**, precision falls. Same story as L03 — the threshold is a business decision.

## Step 7 — Visualise the comparison

In [ ]:
# F1 comparison at the two thresholds
plot_df = comparison_df.set_index(["Model", "Threshold"])["F1"].unstack()

fig, ax = plt.subplots(figsize=(11, 5))
plot_df.plot(kind="bar", ax=ax, color=["steelblue", "coral"], edgecolor="white")
ax.set_ylabel("F1 score")
ax.set_xlabel("")
ax.set_title("F1 comparison — three models at two thresholds")
ax.legend(title="Threshold")
ax.set_ylim(0, max(0.4, plot_df.values.max() * 1.2))
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", padding=3, fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Confusion matrices for each model at threshold 0.25
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (name, model) in zip(axes, [
    ("LR (L03)", lr_pipeline),
    ("RF (tuned)", rf_search.best_estimator_),
    ("GB (tuned)", gb_search.best_estimator_),
]):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.25).astype(int)
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["stayed", "churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"{name} @ threshold 0.25")
plt.tight_layout()
plt.show()

## Step 8 — Sarah's Friday recommendation

| Question | Answer |
|---|---|
| **Highest test F1 (at threshold 0.25)** | Gradient Boosting (tuned) |
| **Easiest to support in production** | Random Forest (fewer hyperparameters to maintain) |
| **Most interpretable** | Logistic Regression — but it's also the weakest performer |
| **Operational fit (~200 calls/week capacity)** | Whichever ranks customers best in the top 200 — usually GB |

Sarah's pitch:

> *"Gradient Boosting tuned is the highest-F1 model — about a 0.04 lift over the L03 baseline at the same threshold. Random Forest is 80% of the way there with less tuning sensitivity. I recommend Gradient Boosting for production. If the engineering team wants minimal hyperparameter maintenance, Random Forest is a defensible second choice. Either way, we're meaningfully ahead of the L03 baseline."*

Marcus nods. *"OK. Ship the gradient boosted model. Now — what about customers who DON'T churn but also don't log in for a year? Can we find natural CLUSTERS of customer behaviour without labels?"*

That question — **finding structure without a target** — is the engine of **L05 (Unsupervised Learning)**.

## ✅ Section Summary

| Step | Output |
|---|---|
| **Random Forest tuned** | Best CV F1 ≈ 0.27-0.31 — tunings around `n_estimators=200`, `min_samples_leaf=5` |
| **Gradient Boosting tuned** | Best CV F1 ≈ 0.31-0.34 — tunings around `learning_rate=0.05`, `max_depth=4` |
| **Test-set F1 (at threshold 0.25)** | GB usually leads RF, both beat LR |
| **The right answer for Friday** | Gradient Boosting tuned — modest but real improvement |

**Key insights:**
- **GridSearchCV is the standard way to tune.** Keep the grid small enough to finish in a sensible time. `n_jobs=-1` is critical.
- **`class_weight='balanced'`** matters more than any single hyperparameter on imbalanced data.
- **Top configurations cluster tightly** — that's a sign the model is robust, not that the search is broken.
- **The Friday recommendation is a judgement call** — F1 isn't the only criterion.

**Next step → `assignment.ipynb`** to apply the L04 toolkit to a Kaggle-style competition on a held-out NorthStar churn scoring set.

*Or open `optional_extensions.ipynb` for Ridge/Lasso regularisation, Gini vs Entropy splits, and RandomizedSearchCV vs GridSearchCV.*

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

## Extension 1 — RandomizedSearchCV: faster than GridSearchCV

When the grid is too big to enumerate, `RandomizedSearchCV` samples N combinations randomly. In practice it finds combinations close to the grid optimum at a fraction of the compute.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Same GB pipeline as before
gb_random_grid = {
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
    "model__max_iter":      [100, 200, 400, 800],
    "model__max_depth":     [3, 4, 5, 6, 8, None],
    "model__min_samples_leaf": [10, 20, 50, 100],
}
# Full grid would be 6×4×6×4 = 576. We sample 25.

start = time.time()
gb_random = RandomizedSearchCV(
    gb_pipeline,
    param_distributions=gb_random_grid,
    n_iter=25,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    random_state=42,
)
gb_random.fit(X_train, y_train)
elapsed = time.time() - start

print(f"RandomizedSearchCV: {elapsed:.1f}s, {len(gb_random.cv_results_['params'])} configurations sampled")
print()
print(f"GridSearchCV    (27 fits): best CV F1 = {gb_search.best_score_:.3f}")
print(f"RandomizedSearch (25 fits): best CV F1 = {gb_random.best_score_:.3f}")
print()
print("RandomizedSearch with a similar fit budget usually lands within 0.01 F1 of GridSearch.")

## Extension 2 — Stability across random_state

How much does the result depend on the random seed? Let's run the tuned GB pipeline with 5 different seeds and see the spread.

In [ ]:
best_params_gb = {k.replace("model__", ""): v for k, v in gb_search.best_params_.items()}

seeds = [42, 7, 123, 999, 2024]
seed_f1 = []
for seed in seeds:
    pipe = Pipeline([
        ("prep",  preprocessor),
        ("model", HistGradientBoostingClassifier(
            **best_params_gb,
            class_weight="balanced",
            random_state=seed)),
    ])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.25).astype(int)
    seed_f1.append((seed, f1_score(y_test, pred)))

seed_df = pd.DataFrame(seed_f1, columns=["random_state", "test_F1_at_0.25"])
print(seed_df.to_string(index=False))
print()
print(f"Mean test F1: {seed_df['test_F1_at_0.25'].mean():.3f}")
print(f"Std test F1:  {seed_df['test_F1_at_0.25'].std():.3f}")
print()
print("A small std (< 0.01) means the model is stable across seeds. A large std means")
print("you got lucky/unlucky with whatever seed you tuned with, and the true performance is uncertain.")

## Extension 3 — Going further with XGBoost

If you have XGBoost installed (`pip install xgboost`), drop it in as the model step. The API is sklearn-compatible — minimal code changes.

In [ ]:
try:
    from xgboost import XGBClassifier

    xgb_pipeline = Pipeline([
        ("prep",  preprocessor),
        ("model", XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
            n_jobs=-1,
            random_state=42,
            eval_metric="logloss",
        )),
    ])
    cv_f1_xgb = cross_val_score(xgb_pipeline, X_train, y_train, cv=cv, scoring="f1", n_jobs=-1)
    print(f"XGBoost (default hyperparameters):  CV F1 = {cv_f1_xgb.mean():.3f} ± {cv_f1_xgb.std():.3f}")
    print(f"sklearn GB (tuned):                 CV F1 = {gb_search.best_score_:.3f}")
    print()
    print("XGBoost and HistGradientBoosting are competitive on most datasets.")
    print("Pick based on team familiarity + infrastructure, not on tiny F1 differences.")
except ImportError:
    print("XGBoost not installed. (pip install xgboost to enable.)")